# Modelling tabular data with diffusion models

This tutorial demonstrates hot to use a denoising diffusion probabilistic model (DDPM) to synthesize tabular data. The algorithm was proposed in [TabDDPM: Modelling Tabular Data with Diffusion Models](https://arxiv.org/abs/2209.15421).

In [ ]:
# stdlib
import sys
import warnings

# third party
import numpy as np
from sklearn.datasets import load_iris, load_diabetes

# synthcity absolute
import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")
warnings.filterwarnings("ignore")

## Synthesize a classification dataset

For classification datasets, TabDDPM automatically uses the labels as the conditional variable during training. You should not provide an additional `cond` argument to the `fit` method.

In [ ]:
# Note: preprocessing data with OneHotEncoder or StandardScaler is not needed or recommended. Synthcity handles feature encoding and standardization internally.

X, y = load_iris(return_X_y=True, as_frame=True)
X["target"] = y

loader = GenericDataLoader(X, target_column="target", sensitive_columns=[])

loader.dataframe().head()

In [ ]:
y.value_counts()

### Model fitting

In [ ]:
# define the model hyper-parameters
plugin_params = dict(
    is_classification = True,
    n_iter = 1000,  # epochs
    lr = 0.002,
    weight_decay = 1e-4,
    batch_size = 1000,
    model_type = "mlp",  # or "resnet"
    model_params = dict(
        n_layers_hidden = 3,
        n_units_hidden = 256,
        dropout = 0.0,
    ),
    num_timesteps = 500,  # timesteps in diffusion
    dim_embed = 128,
    # performance logging
    log_interval = 10,
    print_interval = 100,
)

plugin = Plugins().get("ddpm", **plugin_params)
plugin.fit(loader)

In [ ]:
plugin.model

In [ ]:
# plot training curves
plugin.loss_history.plot()

### Data generation

Since the model training is conditional to the labels, the data generation requires the labels as well. You can pass the labels as a `cond` argument to the `generate` method. If it is not provided, the model will randomly generate the labels following the multinomial distribution of the training labels.

In [ ]:
plugin.generate(10)

### Conditional data generation

In [ ]:
labels = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])
plugin.generate(len(labels), cond=labels)

## Synthesize a regression dataset

For regression datasets, there is no conditional variable by default. The model learns the joint distribution of the whole dataset and generates new data points from it.

In [ ]:
import pandas as pd

df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv", sep=";")

loader = GenericDataLoader(df, target_column="quality", sensitive_columns=[])
loader.dataframe().describe()

In [ ]:
# define the model hyper-parameters
plugin_params.update(
    is_classification = False,
    n_iter = 500,  # epochs
    lr = 5e-4,
    weight_decay = 1e-4,
    batch_size = 1250,
    model_params = dict(
        n_layers_hidden = 3,
        n_units_hidden = 256,
        dropout = 0.0,
    ),
    num_timesteps = 100,  # timesteps in diffusion
)
plugin = Plugins().get("ddpm", **plugin_params)
plugin.fit(loader)

In [ ]:
plugin.loss_history.plot()

In [ ]:
plugin.generate(8)

### Conditional data generation

A conditional variable `cond` can be provided to the `fit` method. It can be either a column name in the dataset or a custom array. The model will then learn the conditional distribution of the dataset given `cond`. In this case, an array must be provided as the `cond` argument of the `generate` method.

Use a column name as the `cond` argument in the `fit` method:

In [ ]:
plugin.fit(loader, cond='quality')

In [ ]:
plugin.loss_history.plot()

In [ ]:
outcome = np.array([3, 4, 5, 6, 7, 8, 9])
outcome

In [ ]:
plugin.generate(len(outcome), cond=outcome)

Use an array as the `cond` argument of the `fit` method:

In [ ]:
import random
cond = random.choices(outcome, k=len(loader))
plugin.fit(loader, cond=cond)

In [ ]:
plugin.loss_history.plot()

In [ ]:
plugin.generate(len(outcome), cond=outcome)

## Congratulations!

Congratulations on completing this notebook tutorial! If you enjoyed this and would like to join the movement towards Machine learning and AI for medicine, you can do so in the following ways!

### Star [Synthcity](https://github.com/vanderschaarlab/synthcity) on GitHub

- The easiest way to help our community is just by starring the Repos! This helps raise awareness of the tools we're building.


### Checkout other projects from vanderschaarlab
- [HyperImpute](https://github.com/vanderschaarlab/hyperimpute)
- [AutoPrognosis](https://github.com/vanderschaarlab/autoprognosis)
